In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/nhattruongdev/musan-noise/musan/README
/kaggle/input/datasets/nhattruongdev/musan-noise/musan/noise/README
/kaggle/input/datasets/nhattruongdev/musan-noise/musan/noise/sound-bible/noise-sound-bible-0017.wav
/kaggle/input/datasets/nhattruongdev/musan-noise/musan/noise/sound-bible/noise-sound-bible-0023.wav
/kaggle/input/datasets/nhattruongdev/musan-noise/musan/noise/sound-bible/noise-sound-bible-0019.wav
/kaggle/input/datasets/nhattruongdev/musan-noise/musan/noise/sound-bible/noise-sound-bible-0021.wav
/kaggle/input/datasets/nhattruongdev/musan-noise/musan/noise/sound-bible/noise-sound-bible-0057.wav
/kaggle/input/datasets/nhattruongdev/musan-noise/musan/noise/sound-bible/noise-sound-bible-0003.wav
/kaggle/input/datasets/nhattruongdev/musan-noise/musan/noise/sound-bible/noise-sound-bible-0028.wav
/kaggle/input/datasets/nhattruongdev/musan-noise/musan/noise/sound-bible/noise-sound-bible-0070.wav
/kaggle/input/datasets/nhattruongdev/musan-noise/musan/noise/sound-bib

In [2]:
!pip install -q soundfile librosa scipy

import os
import re
import glob
import random
import subprocess
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from scipy.signal import butter, sosfilt
from tqdm import tqdm

random.seed(42)
np.random.seed(42)

In [3]:
# TODO: این مسیرها رو با مسیر واقعی input خودتون روی کگل جایگزین کنید
SHEMO_ROOT = "/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database"
MUSAN_ROOT = "/kaggle/input/datasets/nhattruongdev/musan-noise/musan/noise"  # فقط زیرپوشه‌ی noise

OUTPUT_DIR = "/kaggle/working/shemo_augmented"
os.makedirs(OUTPUT_DIR, exist_ok=True)

shemo_files = glob.glob(os.path.join(SHEMO_ROOT, "**", "*.wav"), recursive=True)
print(f"تعداد فایل‌های ShEMO پیدا شده: {len(shemo_files)}")
print(shemo_files[:5])

تعداد فایل‌های ShEMO پیدا شده: 3000
['/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database/female/F13A20.wav', '/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database/female/F23A06.wav', '/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database/female/F24N09.wav', '/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database/female/F24S05.wav', '/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database/female/F21W01.wav']


In [4]:
EMOTION_CODE_MAP = {
    "A": "ANGRY",
    "H": "HAPPY",
    "N": "NEUTRAL",
    "S": "SAD",
    "F": "FEAR",
    "W": "SURPRISE",
}

records = []
for fpath in shemo_files:
    fname = os.path.basename(fpath).replace(".wav", "")
    gender = fname[0]
    speaker_id = fname[1:3]
    emo_code = fname[3]
    if emo_code not in EMOTION_CODE_MAP:
        continue
    records.append({
        "path": fpath,
        "gender": gender,
        "speaker_id": speaker_id,
        "emotion_code": emo_code,
        "emotion": EMOTION_CODE_MAP[emo_code],
    })

shemo_df = pd.DataFrame(records)
print(shemo_df["emotion"].value_counts())
shemo_df.head()

emotion
ANGRY       1059
NEUTRAL     1028
SAD          449
SURPRISE     225
HAPPY        201
FEAR          38
Name: count, dtype: int64


,path,gender,speaker_id,emotion_code,emotion
0,/kaggle/input/datasets/mansourehk/shemo-persia...,F,13,A,ANGRY
1,/kaggle/input/datasets/mansourehk/shemo-persia...,F,23,A,ANGRY
2,/kaggle/input/datasets/mansourehk/shemo-persia...,F,24,N,NEUTRAL
3,/kaggle/input/datasets/mansourehk/shemo-persia...,F,24,S,SAD
4,/kaggle/input/datasets/mansourehk/shemo-persia...,F,21,W,SURPRISE


In [5]:
TARGET_CLASSES = ["ANGRY", "HAPPY", "SAD", "NEUTRAL"]

shemo_filtered = shemo_df[shemo_df["emotion"].isin(TARGET_CLASSES)].reset_index(drop=True)
print(f"تعداد نمونه بعد از فیلتر: {len(shemo_filtered)}")
print(shemo_filtered["emotion"].value_counts())

تعداد نمونه بعد از فیلتر: 2737
emotion
ANGRY      1059
NEUTRAL    1028
SAD         449
HAPPY       201
Name: count, dtype: int64


In [6]:
TARGET_SR = 16000

def load_wav(path, sr=TARGET_SR):
    audio, _ = librosa.load(path, sr=sr, mono=True)
    return audio

musan_noise_files = glob.glob(os.path.join(MUSAN_ROOT, "**", "*.wav"), recursive=True)
print(f"تعداد فایل‌های نویز MUSAN: {len(musan_noise_files)}")
assert len(musan_noise_files) > 0, "فایل نویز پیدا نشد؛ مسیر MUSAN_ROOT رو چک کن"

print("در حال بارگذاری نویزهای MUSAN در حافظه...")
MUSAN_NOISE_CACHE = []
for f in tqdm(musan_noise_files):
    try:
        MUSAN_NOISE_CACHE.append(load_wav(f))
    except Exception:
        continue
print(f"تعداد نویزهای کش‌شده: {len(MUSAN_NOISE_CACHE)}")

تعداد فایل‌های نویز MUSAN: 930
در حال بارگذاری نویزهای MUSAN در حافظه...


100%|██████████| 930/930 [00:25<00:00, 36.13it/s]


تعداد نویزهای کش‌شده: 930


In [7]:
codec_stats = {"success": 0, "fallback": 0}


def add_background_noise(audio, snr_db_range=(10, 25), hard_case_prob=0.1):
    """نویز پس‌زمینه از کش MUSAN؛ فقط درصد کمی نمونه‌ها SNR خیلی سخت می‌گیرن"""
    noise = random.choice(MUSAN_NOISE_CACHE)

    if len(noise) < len(audio):
        reps = int(np.ceil(len(audio) / len(noise)))
        noise = np.tile(noise, reps)
    start = random.randint(0, len(noise) - len(audio))
    noise = noise[start:start + len(audio)]

    if random.random() < hard_case_prob:
        snr_db = random.uniform(3, 10)  # حالت سخت، به‌ندرت
    else:
        snr_db = random.uniform(*snr_db_range)

    audio_power = np.mean(audio ** 2) + 1e-10
    noise_power = np.mean(noise ** 2) + 1e-10
    target_noise_power = audio_power / (10 ** (snr_db / 10))
    noise = noise * np.sqrt(target_noise_power / noise_power)

    return audio + noise


def apply_reverb(audio, sr=TARGET_SR, decay_range=(0.15, 0.35)):
    """reverb ملایم‌تر و طبیعی‌تر (بدون تکیه‌ی صرف به نویز گاوسی خام)"""
    decay = random.uniform(*decay_range)
    rir_len = int(sr * 0.25)  # کوتاه‌تر از نسخه‌ی قبلی (400ms زیادی بلند بود)
    t = np.linspace(0, 1, rir_len)
    rir = np.exp(-t / decay) * np.random.uniform(0.3, 0.7, rir_len)
    rir = rir / (np.max(np.abs(rir)) + 1e-8)
    reverbed = np.convolve(audio, rir, mode="full")[:len(audio)]
    wet_ratio = random.uniform(0.1, 0.3)  # wet ratio کمتر از قبل
    return (1 - wet_ratio) * audio + wet_ratio * reverbed


def apply_mic_frequency_response(audio, sr=TARGET_SR, cutoff=100):
    """شبیه‌سازی محدودیت فرکانسی میکروفون گوشی (افت باس)"""
    sos = butter(4, cutoff, btype="highpass", fs=sr, output="sos")
    return sosfilt(sos, audio)


def apply_agc_like_gain(audio, gain_db_range=(-6, 6), threshold=0.3, ratio=3.0):
    """gain + compressor ساده برای رفتار شبیه AGC گوشی"""
    gain_db = random.uniform(*gain_db_range)
    gain_factor = 10 ** (gain_db / 20)
    audio = audio * gain_factor

    compressed = np.copy(audio)
    over_thresh = np.abs(audio) > threshold
    sign = np.sign(audio[over_thresh])
    excess = np.abs(audio[over_thresh]) - threshold
    compressed[over_thresh] = sign * (threshold + excess / ratio)

    return compressed


def simulate_recording_codec(audio, sr=TARGET_SR, codec="aac"):
    """شبیه‌سازی کدک ضبط گوشی: AAC یا Opus (بدون AMR، چون مخصوص مکالمه‌ی تلفنیه)"""
    tmp_in = "/tmp/tmp_in.wav"
    ext = "m4a" if codec == "aac" else "opus"
    tmp_codec = f"/tmp/tmp_codec.{ext}"
    tmp_out = "/tmp/tmp_out.wav"

    sf.write(tmp_in, audio, sr)

    if codec == "aac":
        subprocess.run(
            ["ffmpeg", "-y", "-i", tmp_in, "-ar", str(sr), "-ac", "1", "-c:a", "aac", "-b:a", "64k", tmp_codec],
            capture_output=True,
        )
    else:
        subprocess.run(
            ["ffmpeg", "-y", "-i", tmp_in, "-ar", str(sr), "-ac", "1", "-c:a", "libopus", "-b:a", "32k", tmp_codec],
            capture_output=True,
        )

    subprocess.run(["ffmpeg", "-y", "-i", tmp_codec, "-ar", str(sr), tmp_out], capture_output=True)

    if not os.path.exists(tmp_out):
        codec_stats["fallback"] += 1
        return audio

    out_audio, _ = librosa.load(tmp_out, sr=sr, mono=True)
    if len(out_audio) > len(audio):
        out_audio = out_audio[:len(audio)]
    else:
        out_audio = np.pad(out_audio, (0, len(audio) - len(out_audio)))

    codec_stats["success"] += 1
    return out_audio

In [8]:
!ffmpeg -codecs 2>/dev/null | grep -i aac
!ffmpeg -codecs 2>/dev/null | grep -i opus

 DEAIL. aac                  AAC (Advanced Audio Coding) (decoders: aac aac_fixed )
 D.AIL. aac_latm             AAC LATM (Advanced Audio Coding LATM syntax)
 D.VI.S cllc                 Canopus Lossless Codec
 D.VIL. hq_hqa               Canopus HQ/HQA
 D.VIL. hqx                  Canopus HQX
 DEAIL. opus                 Opus (Opus Interactive Audio Codec) (decoders: opus libopus ) (encoders: opus libopus )


In [9]:
def augment_pipeline(audio, sr=TARGET_SR):
    aug = audio.copy()

    # ۱. نویز محیط (قبل از رسیدن به میکروفون)
    if random.random() < 0.7:
        aug = add_background_noise(aug)

    # ۲. پیچش صدا در فضا
    if random.random() < 0.5:
        aug = apply_reverb(aug, sr=sr)

    # ۳. محدودیت فرکانسی میکروفون گوشی
    if random.random() < 0.6:
        aug = apply_mic_frequency_response(aug, sr=sr)

    # ۴. gain + AGC-like compression
    if random.random() < 0.7:
        aug = apply_agc_like_gain(aug)

    # نرمال‌سازی قبل از کدک (جلوگیری از clipping در انکود)
    max_val = np.max(np.abs(aug)) + 1e-8
    if max_val > 1.0:
        aug = aug / max_val

    # ۵. فشرده‌سازی نهایی با کدک ضبط (آخرین مرحله، روی سیگنال نهایی)
    if random.random() < 0.6:
        codec = random.choice(["aac", "opus"])
        aug = simulate_recording_codec(aug, sr=sr, codec=codec)

    max_val = np.max(np.abs(aug)) + 1e-8
    if max_val > 1.0:
        aug = aug / max_val

    return aug

In [10]:
BASE_AUG_MULTIPLIER = 2  # همه‌ی کلاس‌ها حداقل این‌قدر augment می‌گیرن

class_counts = shemo_filtered["emotion"].value_counts()
max_count = class_counts.max()

aug_per_sample_by_class = {}
for emo, count in class_counts.items():
    balance_boost = max(0, round((max_count - count) / count))
    aug_per_sample_by_class[emo] = BASE_AUG_MULTIPLIER + balance_boost

print("تعداد augmentation به ازای هر نمونه، به تفکیک کلاس:")
print(aug_per_sample_by_class)

print("\nحجم تخمینی هر کلاس بعد از augmentation:")
for emo, count in class_counts.items():
    total_after = count + count * aug_per_sample_by_class[emo]
    print(f"  {emo}: {count} -> {total_after}")

تعداد augmentation به ازای هر نمونه، به تفکیک کلاس:
{'ANGRY': 2, 'NEUTRAL': 2, 'SAD': 3, 'HAPPY': 6}

حجم تخمینی هر کلاس بعد از augmentation:
  ANGRY: 1059 -> 3177
  NEUTRAL: 1028 -> 3084
  SAD: 449 -> 1796
  HAPPY: 201 -> 1407


In [11]:
augmented_records = []
skipped_silent = 0

for _, row in tqdm(shemo_filtered.iterrows(), total=len(shemo_filtered)):
    audio = load_wav(row["path"])
    n_aug = aug_per_sample_by_class[row["emotion"]]

    for i in range(n_aug):
        aug_audio = augment_pipeline(audio)

        # کنترل کیفیت: اگه صدا عملاً از بین رفته (خیلی نادره ولی محض احتیاط)
        if np.max(np.abs(aug_audio)) < 0.01:
            skipped_silent += 1
            continue

        out_fname = f"{os.path.basename(row['path']).replace('.wav', '')}_aug{i}.wav"
        out_path = os.path.join(OUTPUT_DIR, out_fname)
        sf.write(out_path, aug_audio, TARGET_SR)

        augmented_records.append({
            "path": out_path,
            "original_path": row["path"],
            "speaker_id": row["speaker_id"],
            "gender": row["gender"],
            "emotion": row["emotion"],
            "is_augmented": True,
        })

augmented_df = pd.DataFrame(augmented_records)
print(f"\nتعداد نمونه‌های augmented تولیدشده: {len(augmented_df)}")
print(f"تعداد نمونه‌های رد شده (تقریباً بی‌صدا): {skipped_silent}")
print(f"آمار موفقیت کدک: {codec_stats}")

if codec_stats["success"] == 0 and codec_stats["fallback"] > 0:
    print("⚠️ هشدار: هیچ‌کدوم از کدک‌ها موفق نشدن — augmentation کدک عملاً بی‌اثر بوده!")

100%|██████████| 2737/2737 [22:34<00:00,  2.02it/s]


تعداد نمونه‌های augmented تولیدشده: 6727
تعداد نمونه‌های رد شده (تقریباً بی‌صدا): 0
آمار موفقیت کدک: {'success': 3981, 'fallback': 0}


In [12]:
shemo_filtered["is_augmented"] = False
shemo_filtered["original_path"] = shemo_filtered["path"]

final_df = pd.concat([
    shemo_filtered[["path", "original_path", "speaker_id", "gender", "emotion", "is_augmented"]],
    augmented_df,
], ignore_index=True)

manifest_path = "/kaggle/working/shemo_augmented_manifest.csv"
final_df.to_csv(manifest_path, index=False)

print(final_df["emotion"].value_counts())
print(f"مجموع نمونه (اصلی + augmented): {len(final_df)}")
print(f"Manifest ذخیره شد در: {manifest_path}")
final_df.head()

emotion
ANGRY      3177
NEUTRAL    3084
SAD        1796
HAPPY      1407
Name: count, dtype: int64
مجموع نمونه (اصلی + augmented): 9464
Manifest ذخیره شد در: /kaggle/working/shemo_augmented_manifest.csv


,path,original_path,speaker_id,gender,emotion,is_augmented
0,/kaggle/input/datasets/mansourehk/shemo-persia...,/kaggle/input/datasets/mansourehk/shemo-persia...,13,F,ANGRY,False
1,/kaggle/input/datasets/mansourehk/shemo-persia...,/kaggle/input/datasets/mansourehk/shemo-persia...,23,F,ANGRY,False
2,/kaggle/input/datasets/mansourehk/shemo-persia...,/kaggle/input/datasets/mansourehk/shemo-persia...,24,F,NEUTRAL,False
3,/kaggle/input/datasets/mansourehk/shemo-persia...,/kaggle/input/datasets/mansourehk/shemo-persia...,24,F,SAD,False
4,/kaggle/input/datasets/mansourehk/shemo-persia...,/kaggle/input/datasets/mansourehk/shemo-persia...,24,F,SAD,False


In [13]:
import IPython.display as ipd

N_SAMPLES = 10
sample_rows = augmented_df.sample(N_SAMPLES, random_state=42)

for idx, sample_row in sample_rows.iterrows():
    print(f"--- نمونه {idx} | احساس: {sample_row['emotion']} | فایل: {os.path.basename(sample_row['path'])} ---")

    print("نسخه‌ی augmented:")
    display(ipd.Audio(sample_row["path"]))

    print("نسخه‌ی اصلی:")
    display(ipd.Audio(sample_row["original_path"]))

    print()

--- نمونه 5022 | احساس: NEUTRAL | فایل: M12N63_aug0.wav ---
نسخه‌ی augmented:


نسخه‌ی اصلی:



--- نمونه 132 | احساس: HAPPY | فایل: F06H01_aug4.wav ---
نسخه‌ی augmented:


نسخه‌ی اصلی:



--- نمونه 5402 | احساس: NEUTRAL | فایل: M49N04_aug1.wav ---
نسخه‌ی augmented:


...

نسخه‌ی اصلی:



--- نمونه 3559 | احساس: NEUTRAL | فایل: M21N10_aug0.wav ---
نسخه‌ی augmented:


نسخه‌ی اصلی:



--- نمونه 2333 | احساس: SAD | فایل: F06S09_aug1.wav ---
نسخه‌ی augmented:


نسخه‌ی اصلی:
